# Programowanie i metody numeryczne

### Ćwiczenia 11.
### Równania różniczkowe zwyczajne - część 1.

---

In [ ]:
import numpy as np
import scipy.integrate
import scipy.special
import matplotlib.pyplot as plt
%matplotlib inline

---

## Równania różniczkowe zwyczajne: zagadnienie początkowe

SciPy udostępnia funkcję do numerycznego rozwiązywania zagadnień początkowych: `scipy.integrate.solve_ivp`.

---

### Zadanie 1. Oscylator harmoniczny.

Zaczniemy od oscylatora harmonicznego, a więc układu, dla którego znamy rozwiązanie analityczne - pozwoli nam to łatwo ocenić jakość różnych metod całkowania zagadnień początkowych.

Równanie ruchu oscylatora harmonicznego o częstości $\omega $ ma postać
$$
\frac{\mathrm{d}^2 x}{\mathrm{d} t^2} + \omega ^2 x = 0 ,
$$
gdzie $x = x(t)$ jest zmienną dynamiczną opisującą wychylenie oscylatora z położenia równowagi w funkcji czasu $t$. Równanie to należy do klasy liniowych jednorodnych równań różniczkowych zwyczajnych drugiego rzędu.

Przyjmujemy warunki początkowe
$$
x(0) = x_0, \qquad v(0) = v_0 .
$$

Znane jest analityczne rozwiązanie równania ruchu oscylatora harmonicznego. Można je przedstawić w postaci
$$
x(t) = A \sin \omega t + B \cos \omega t,
$$
gdzie stałe $A$ i $B$ wyznaczone są przez warunku początkowe:
$$
A = \frac{v_0}{\omega}, \qquad
B = x_0.
$$

In [ ]:
def solve_exact(
    omega: float,
    x0:    float,
    v0:    float,
    t:     float
) -> float:
    """Oscylator harmoniczny - rozwiązanie analityczne."""
    A = v0 / omega
    B = x0
    return A * np.sin(omega * t) + B * np.cos(omega * t)

Równanie oscylatora jest równaniem rzędu wyższego niż pierwszy, zatem pierwszym krokiem rozwiązania numerycznego będzie zastąpienie tego równania równoważnym układem równań pierwszego rzędu. Wprowadzając zmienną
$$
v(t) \equiv \frac{\mathrm{d} x}{\mathrm{d} t}
$$
otrzymamy układ
$$
\begin{cases}
\displaystyle \frac{\mathrm{d} x}{\mathrm{d} t} = v, \\[5px]
\displaystyle \frac{\mathrm{d} v}{\mathrm{d} t} = - \omega ^2 x .
\end{cases}
$$
Właśnie ten układ równań różniczkowych zwyczajnych pierwszego rzędu będziemy rozwiązywać numerycznie.

**Funkcje pomocnicze**.

* Prawa strona pierwszego równania.

In [ ]:
def f_x(v: float) -> float:
    """Pochodna x (prawa strona pierwszego równania)."""
    return v

* Prawa strona drugiego równania.

In [ ]:
def f_v(omega: float, x: float) -> float:
    """Pochodna v (prawa strona drugiego równania)."""
    return - (omega ** 2) * x

* Wizualizacja wyników.

In [ ]:
def plot_harmonic_oscillator(
    res_t:     list[float],
    res_x:     list[float],
    res_v:     list[float],
    res_exact: list[float],
    res_diff:  list[float],
    method:    str
) -> None:
    """Wykresy rozwiązań numerycznego i analitycznego."""
    fig = plt.figure(figsize=(12,8), dpi= 100,constrained_layout = True)

    fig.suptitle(f"Oscylator harmoniczny – metoda {method}", fontsize = "xx-large")

    subfigs = fig.subfigures(1, 2, wspace=0.07)

    subfigs[1].suptitle("Diagram fazowy")

    axs = subfigs[0].subplots(2)
    axs[0].plot(res_t, res_x, 'r-', label = "Rozwiązanie numeryczne")
    axs[0].plot(res_t, res_exact, 'b-', label = "Rozwiązanie analityczne")
    axs[0].set_title("Rozwiązania: numeryczne (czerwone) i analityczne (niebieskie)")
    axs[0].grid()

    axs[1].plot(res_t, res_diff, 'g-')
    axs[1].set_title("Różnica pomiędzy rozwiązaniem analitycznym a numerycznym")
    axs[1].grid()

    for ax in axs.flat:
        ax.set(xlabel = "t", ylabel = "x")

    for ax in axs.flat:
        ax.label_outer()
        
    axr = subfigs[1].subplots()
    axr.plot(res_x, res_v)
    axr.grid()
    axr.set(xlabel = "x", ylabel = "v")

    plt.show()
    

**Metoda Eulera**.

In [ ]:
def solve_euler(
    omega: float,
    x0:    float,
    v0:    float,
    tmax:  float,
    dt:    float
) -> tuple[list[float], list[float], list[float], list[float], list[float]]:
    """Oscylator harmoniczny - metoda Eulera.
    Argumenty:
        omega: częstość oscylatora,
        x0:    położenie początkowe,
        v0:    prędkość początkowa,
        tmax:  czas końcowy, do którego będziemy prowadzić obliczenia,
        dt:    krok czasowy.
    Zwraca:
        res_t:     listę kolejnych chwil, różniących się od siebie o dt,
        res_x:     listę położeń oscylatora w kolejnych chwilach,
        res_v:     listę prędkości oscylatora w kolejnych chwilach,
        res_exact: listę analitycznych wartości położenia oscylatora w kolejnych chwilach,
        res_diff:  listę różnic pomiędzy wartościami analitycznymi a numerycznymi."""
    
    # Zmienne x i v będą przechowywały, odpowiednio, położenie i prędkość oscylatora
    # w danej iteracji. Na początek przypisujemy im więc wartości początkowe.
    x, v  =  x0, v0

    # Tworzymy listy, które będą przechowywały wyniki obliczeń uzyskane w kolejnych
    # iteracjach. Kolejne elementy list będą sobie wzajemnie odpowiadały, np. res_x[3]
    # będzie położeniem oscylatora w chwili res_t[3], zaś np. res_v[5] - prędkością
    # oscylatora w chwili res_t[5]. Na listach umieszczamy od razu wartości początkowe.

    # res_t - kolejne chwile, różniące się od siebie o dt.
    res_t = [0.]

    # res_x - położenia oscylatora w kolejnych chwilach.
    res_x = [x0]

    # res_v - prędkości oscylatora w kolejnych chwilach.
    res_v = [v0]

    # Dla zobraozwania dokładności metody numerycznej, którą stosujemy, obliczymy też
    # dokładne wartości położenia oscylatora w kolejnych chwilach, uzyskane na podstawie
    # rozwiązania analitycznego, oraz różnicę pomiędzy wartością dokładną a numeryczną.

    # res_exact - położenia oscylatora w kolejnych chwilach - wynik analityczny.
    res_exact = [x0]

    # res_diff - różnica pomiędzy wynikiem analitycznym a numerycznym.
    res_diff  = [0.]

    # Główna pętla przeprowadzająca kolejne iteracje algorytmu.
    for t in np.arange(dt, tmax, dt):
        # Obliczamy nowe wartości położenia i prędkości.
        new_x = x + dt * f_x(v)
        new_v = v + dt * f_v(omega, x)

        # Dopisujemy te wartości do list zawierających wyniki.
        res_t.append(t)
        res_x.append(new_x)
        res_v.append(new_v)

        # Obliczamy też wartość analityczną dla danej chwili oraz różnicę pomiędzy
        # wartościami analityczną i numeryczną oraz dopisujemy je do odpowiednich list.
        x_exact = solve_exact(omega, x0, v0, t)
        res_exact.append(x_exact)
        res_diff.append(x_exact - x)

        # Przygotowujemy się do kolejnej iteracji, aktualizując zmienne x i v.
        x, v = new_x, new_v
   
    # Po wykonaniu wszystkich iteracji zwracamy otrzymane wyniki.
    return res_t, res_x, res_v, res_exact, res_diff

Parametry sumulacji. Poeksperymentuj z różnymi wartościami.

In [ ]:
omega = 1.
x0    = 1.
v0    = 0.
tmax  = 100.
dt    = 0.01

In [ ]:
plot_harmonic_oscillator(*solve_euler(omega, x0, v0, tmax, dt), method = "Eulera")

**Meetoda punktu środkowego**.

In [ ]:
def solve_midpoint(
    omega: float,
    x0:    float,
    v0:    float,
    tmax:  float,
    dt:    float
) -> tuple[list[float], list[float], list[float], list[float], list[float]]:
    """Oscylator harmoniczny - metoda punktu środkowego.
    Argumenty:
        omega: częstość oscylatora,
        x0:    położenie początkowe,
        v0:    prędkość początkowa,
        tmax:  czas końcowy, do którego będziemy prowadzić obliczenia,
        dt:    krok czasowy.
    Zwraca:
        res_t:     listę kolejnych chwil, różniących się od siebie o dt,
        res_x:     listę położeń oscylatora w kolejnych chwilach,
        res_v:     listę prędkości oscylatora w kolejnych chwilach,
        res_exact: listę analitycznych wartości położenia oscylatora w kolejnych chwilach,
        res_diff:  listę różnic pomiędzy wartościami analitycznymi a numerycznymi."""
    
    # Zmienne x i v będą przechowywały, odpowiednio, położenie i prędkość oscylatora
    # w danej iteracji. Na początek przypisujemy im więc wartości początkowe.
    x, v  =  x0, v0

    # Tworzymy listy, które będą przechowywały wyniki obliczeń uzyskane w kolejnych
    # iteracjach. Kolejne elementy list będą sobie wzajemnie odpowiadały, np. res_x[3]
    # będzie położeniem oscylatora w chwili res_t[3], zaś np. res_v[5] - prędkością
    # oscylatora w chwili res_t[5]. Na listach umieszczamy od razu wartości początkowe.

    # res_t - kolejne chwile, różniące się od siebie o dt.
    res_t = [0.]

    # res_x - położenia oscylatora w kolejnych chwilach.
    res_x = [x0]

    # res_v - prędkości oscylatora w kolejnych chwilach.
    res_v = [v0]

    # Dla zobraozwania dokładności metody numerycznej, którą stosujemy, obliczymy też
    # dokładne wartości położenia oscylatora w kolejnych chwilach, uzyskane na podstawie
    # rozwiązania analitycznego, oraz różnicę pomiędzy wartością dokładną a numeryczną.

    # res_exact - położenia oscylatora w kolejnych chwilach - wynik analityczny.
    res_exact = [x0]

    # res_diff - różnica pomiędzy wynikiem analitycznym a numerycznym.
    res_diff  = [0.]

    # Główna pętla przeprowadzająca kolejne iteracje algorytmu.
    for t in np.arange(dt, tmax, dt):
        # Obliczamy nowe wartości położenia i prędkości.
        aux_x = x + dt * f_x(v) / 2.
        aux_v = v + dt * f_v(omega, x) / 2.

        new_x = x + dt * f_x(aux_v)
        new_v = v + dt * f_v(omega, aux_x)

        # Dopisujemy te wartości do list zawierających wyniki.
        res_t.append(t)
        res_x.append(new_x)
        res_v.append(new_v)

        # Obliczamy też wartość analityczną dla danej chwili oraz różnicę pomiędzy
        # wartościami analityczną i numeryczną oraz dopisujemy je do odpowiednich list.
        x_exact = solve_exact(omega, x0, v0, t)
        res_exact.append(x_exact)
        res_diff.append(x_exact - x)

        # Przygotowujemy się do kolejnej iteracji, aktualizując zmienne x i v.
        x, v = new_x, new_v
   
    # Po wykonaniu wszystkich iteracji zwracamy otrzymane wyniki.
    return res_t, res_x, res_v, res_exact, res_diff

Parametry symulacji. Poeksperymentuj z różnymi wartościami.

In [ ]:
omega = 1.
x0    = 1.
v0    = 0.
tmax  = 100.
dt    = 0.01

In [ ]:
plot_harmonic_oscillator(*solve_midpoint(omega, x0, v0, tmax, dt), method = "punktu środkowego")

1. Odpowiednio modyfikując kod powyższych funkcji, napisz funkcję `solve_RK4`, rozwiązującą równanie oscylatora harmonicznego metodą Rungego-Kutty 4. rzędu.

In [ ]:
def solve_RK4(
    omega: float,
    x0:    float,
    v0:    float,
    tmax:  float,
    dt:    float
) -> tuple[list[float], list[float], list[float], list[float], list[float]]:
    """Oscylator harmoniczny - metoda RK4.
    Argumenty:
        omega: częstość oscylatora,
        x0:    położenie początkowe,
        v0:    prędkość początkowa,
        tmax:  czas końcowy, do którego będziemy prowadzić obliczenia,
        dt:    krok czasowy.
    Zwraca:
        res_t:     listę kolejnych chwil, różniących się od siebie o dt,
        res_x:     listę położeń oscylatora w kolejnych chwilach,
        res_v:     listę prędkości oscylatora w kolejnych chwilach,
        res_exact: listę analitycznych wartości położenia oscylatora w kolejnych chwilach,
        res_diff:  listę różnic pomiędzy wartościami analitycznymi a numerycznymi."""
    
    # Miejsce na Twoje rozwiązanie

    return res_t, res_x, res_v, res_exact, res_diff

In [ ]:
omega = 1.
x0    = 1.
v0    = 0.
tmax  = 100.
dt    = 0.01

plot_harmonic_oscillator(*solve_RK4(omega, x0, v0, tmax, dt), method = "RK4")

2. Wykorzystaj różne metody całkowania równania oferowane przez funkcję `solve_ivp` ze SciPy. Narysuj wykresy jak powyżej, stosując funkcję `plot_harmonic_oscillator`. Która z nich radzi sobie najlepiej? Która jest najszybsza?

In [ ]:
# Miejsce na Twoje rozwiązanie

# Możesz dodać więcej komórek z kodem

### Zadanie 2. Wahadło matematyczne.

Równanie ruchu wahadła matematycznego ma postać
$$
\frac{\mathrm{d}^2 \theta }{\mathrm{d} t^2} + \frac{g}{l} \sin \theta = 0 ,
$$
gdzie $g$ jest przyspieszeniem grawitacyjnym, $l$ -- długością wahadła, zaś $\theta = \theta (t)$ -- zmienną dynamiczną opisującą wychylenie wahadła z położenia równowagi w funkcji czasu $t$.

1. Napisz funkcję `solve_pendulum` rozwiązującą równanie ruchu wahadła matematycznego numerycznie (bez stosowania przybliżenia małych drgań). Funkcja powinna wyznaczać wychylenie $\theta $ i prędkość kątową $\omega = \mathrm{d} \theta / \mathrm{d}t$ wahadła o zadanej długości $l$ w przedziale czasowym $t \in [ 0, \, t_\mathrm{max}]$ z krokiem $\delta t$ dla zadanych warunków początkowych $\theta (0) = \theta _0$ i $\omega (0) = \omega _0$.

   Twoja funkcja powinna tworzyć trzy rysunki: wykresy funkcji $\theta = \theta (t)$ i $\omega = \omega (t)$ oraz diagram fazowy $\omega = \omega (\theta )$.

   Wykorzystaj funkcję `scipy.integrate.solve_ivp` i domyślną metodę całkowania.

In [ ]:
# Miejsce na Twoje rozwiązanie

# Możesz dodać więcej komórek z kodem

2. Poeksperymentuj z różnymi metodami całkowania równania. Porównaj ich jakość oraz czas wykonywania obliczeń.

In [ ]:
# Miejsce na Twoje rozwiązanie

# Możesz dodać więcej komórek z kodem

### Zadanie 3. Oscylator Duffinga.

Oscylatorem Duffinga nazywamy układ mechaniczny opisywany równaniem ruchu
$$
\frac{\mathrm{d}^2 x}{\mathrm{d} t^2} + \delta \frac{\mathrm{d} x}{\mathrm{d} t} + \beta x + \alpha x^3 = \gamma \cos (\omega t) .
$$
Wielkość $x = x(t)$ jest zmienną dynamiczną określającą wychylenie oscylatora z położenia równowagi w funkcji czasu $t$, zaś stałe $\alpha $, $\beta $, $\gamma $ i $\delta $ - parametrami. Układ ten wykazuje zachowanie chaotyczne - niewielka zmiana warunków początkowych znacząco wpływa na jego ewolucję.

1. Napisz funkcję `solve_duffing` rozwiązującą numerycznie równanie ruchu oscylatora Duffinga. Funkcja powinna wyznaczać wychylenie $x$ i prędkość $v = \mathrm{d}x / \mathrm{d}t$ oscylatora o zadanych wartościach parametrów
   $$
   \alpha = \omega = 1, \quad \beta = - 1, \quad \delta = 0,2 , \quad \gamma = 0,3
   $$
   w przedziale czasowym $t \in [ 0, \, t_\mathrm{max}]$ z krokiem $\delta t$.
   
   Jako argumenty funkcja powinna przyjmować cztery liczby rzeczywiste reprezentujące kolejno czas trwania symulacji $t_\mathrm{max}$, krok czasowy $\delta t$, położenie początkowe $x_0$ i prędkość początkową $v_0$.
   
   Posłuż się funkcją `scipy.integrate.solve_ivp` i metodą `RK45`.

In [ ]:
# Miejsce na Twoje rozwiązanie

# Możesz dodać więcej komórek z kodem

2. Wykorzystując swoją funkcję, przedstaw na jednym rysunku diagramy fazowe oscylatora, $v = v(x)$, dla trzech zestawów wartości początkowych:
   $$
   \begin{dcases}
   x_{01} = 0,99 \\
   v_{01} = 0,
   \end{dcases}
   \qquad
   \begin{dcases}
   x_{02} = 1 \\
   v_{02} = 0,
   \end{dcases}
   \qquad
   \begin{dcases}
   x_{03} = 1.01 \\
   v_{03} = 0.
   \end{dcases}
   $$
   Jak zmienia się krzywa fazowa przy tak niewielkiej zmianie $x_0$?

In [ ]:
# Miejsce na Twoje rozwiązanie

# Możesz dodać więcej komórek z kodem

### Zadanie 4. Model Lotki-Volterry.

Rozważmy ekosystem złożony z dwóch oddziałujących ze sobą populacji: drapieżników oraz ich ofiar. Niech $t$ będzie czasem, $x = x(t)$ - liczebnością populacji ofiar, zaś $y = y(t)$ - liczebnością populacji drapieżników. Ewolucję czasową tych wielkości opisują (w uproszczeniu) równania Lotki-Volterry:
$$
\begin{dcases}
\frac{\mathrm{d}x}{\mathrm{d}t} &= \, \, \, \, (a - by) \, x , \\[7px]
\frac{\mathrm{d}y}{\mathrm{d}t} &= \, \, \, \, (cx - d) \, y .
\end{dcases}
$$
Stałe parametry $a$, $b$, $c$ i $d$ opisują odpowiednio: naturalny przyrost populacji ofiar, zmniejszanie się populacji ofiar wskutek drapieżnictwa, wzrost populacji drapieżników związany z dostępnością pożywienia oraz naturalne zmniejszanie się populacji drapieżników.

1. Napisz funkcję `solve_lv` rozwiązującą numerycznie równania Lotki-Volterry. Funkcja powinna wyznaczać liczebność populacji ofiar $x$ i liczebność populacji drapieżników $y$ w przedziale czasowym $t \in [ 0, \, t_\mathrm{max}]$ z krokiem $\delta t$ dla zadanych warunków początkowych $x(0) = x_0$ i $y(0) = y_0$ oraz zadanych wartości parametrów $a$, $b$, $c$ i $d$.

   Jako argumenty funkcja przyjmować osiem liczb rzeczywistych reprezentujących kolejno: parametry $a$, $b$, $c$ i $d$, wartości początkowe $x_0$ i $y_0$, czas trwania symulacji $t_\mathrm{max}$ oraz krok czasowy $\delta t$.

   Posłuż się funkcją `scipy.integrate.solve_ivp` i metodą `RK4`.

In [ ]:
# Miejsce na Twoje rozwiązanie

# Możesz dodać więcej komórek z kodem

2. Korzystając ze swojej funkcji, wykonaj rysunek przedstawiający wykresy zależności $x = x(t)$ oraz $y = y(t)$ (w jednym układzie współrzędnych) oraz rysunek prezentujący diagram $y = y(x)$.

In [ ]:
# Miejsce na Twoje rozwiązanie

# Możesz dodać więcej komórek z kodem

---

## Praca domowa

Rozwiąż numerycznie i przedstaw na wykresach ruch następujących układów:

* wahadło podwójne (należy znaleźć ruch każdej z mas tworzących wahdło),

* dwa koraliki nanizane na pionową obręcz obracającą się wokół swojej osi, połączone sprężyną.

Możesz dodać więcej układów, które znasz z mechaniki.